<!-- journey-badges -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NolaNexus/LMp2LMa/blob/main/track-02-dl-programmer/07_nlp_finetune.ipynb)
[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/NolaNexus/LMp2LMa/blob/main/track-02-dl-programmer/07_nlp_finetune.ipynb)
[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/NolaNexus/LMp2LMa)

# NLP Fine-tuning: LoRA / QLoRA on LLM

In [ ]:
# Install required packages
!{sys.executable} -m pip install peft==0.7.0 bitsandbytes>=0.41.0 accelerate==0.25.0 transformers torch datasets -q 2>/dev/null || true

In [ ]:
import torch
from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Small local model that loads quickly
try:
    model_name = "microsoft/phi-2"
except Exception:
    model_name = "meta-llama/Llama-2-7b-hf"

print(
    f"Using model: {model_name}"
)  # Load tokenizer (REQUIRED - previously missing)tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)if tokenizer.pad_token is None:    tokenizer.pad_token = tokenizer.eos_tokenprint(f"Tokenizer loaded: {tokenizer.__class__.__name__}")from datasets import load_dataset

## 1. QLoRA Configuration

QLoRA = 4-bit quantization (NF4) + LoRA adapters
- Enables fine-tuning 7B model on 24GB VRAM GPU
- ~97% parameter reduction vs full training

In [ ]:
# QLoRA config: 4-bit quantization + LoRA adapters
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print(f"Total parameters: {sum(p.numel() for p in model.parameters())}")
print(
    f"Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad) / sum(p.numel() for p in model.parameters()) * 100:.1f}%"
)

## 2. Synthetic Dataset (no external download needed)

Use tiny synthetic instruction dataset instead of HuggingFace download.

In [ ]:
# Build a small instruction dataset locally
data = [
    {
        "instruction": "Write a Python function to calculate factorial",
        "input": "",
        "output": "def factorial(n):\n    if n <= 1: return 1\n    return n * factorial(n-1)",
    },
    {
        "instruction": "What is the area of a circle with radius 5?",
        "input": "",
        "output": "Area = pi * r^2 = pi * 25 = approx 78.54",
    },
    {
        "instruction": "Convert 3 hours to seconds",
        "input": "",
        "output": "3 * 3600 = 10800 seconds",
    },
    {
        "instruction": "Explain gradient descent optimization",
        "input": "",
        "output": "Gradient descent updates parameters by moving opposite to the gradient of the loss function w.r.t. each parameter.",
    },
    {
        "instruction": "Write an SQL query to find top 5 customers by revenue",
        "input": "SELECT customer, SUM(revenue) as total_rev FROM orders GROUP BY customer ORDER BY total_rev DESC LIMIT 5;",
        "output": null,
    },
]

In [ ]:
# Save & load via load_dataset
data_path = "./synth_instructions.json"
import json

with open(data_path, "w") as f:
    json.dump(data, f, indent=2)

dataset = load_dataset("json", data_files={"train": data_path})


def format_instruction(example):
    if example["input"]:  # type: ignore
        return f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n{example['output']}"
    else:
        return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"


dataset = dataset.map(format_instruction)


def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )


dataset = dataset.map(tokenize_function, batched=True)
dataset = dataset.remove_columns(["instruction", "input", "output", "text"])
dataset.set_format("torch")

print(f"Dataset size: {len(dataset)}")
print(f"Classes: {len(dataset.features)}")

## 3. Training Configuration & Execution

Full QLoRA training loop executed here (no longer commented out).

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-finetuned",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    warmup_steps=50,
    learning_rate=5e-4,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

print("Starting QLoRA training...")
results = trainer.train()
print(f"Training done. Eval loss: {results.eval_loss:.4f}")

## 4. Evaluation & Generation

Load fine-tuned adapter and test generation quality.
Note: On CPU the generation may be limited; the real advantage shows on GPU.
On GPU this generates fluent responses quickly.

In [ ]:
# Load fine-tuned adapter for inference
def load_finetuned(base_model_path, adapter_path):
    model = AutoModelForCausalLM.from_pretrained(
        base_model_path,
        quantization_config=bnb_config,
        device_map="auto",
    )
    return PeftModel.from_pretrained(model, adapter_path)


def generate(model, prompt, max_new_tokens=50, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


try:
    model = load_finetuned(model_name, "./lora-finetuned/adapter")
except Exception as e:
    print(f"Warning: merged model not found yet ({e}), skipping inference demo")
    model = model

if hasattr(model, "merge_and_unload"):
    try:
        model = model.merge_and_unload()
        print("Merged model loaded.")
    except Exception:
        pass

prompt = "Write a Python function to calculate"
print("Generated response:")
print(generate(model, prompt, max_new_tokens=80))

## 5. Other PEFT Methods Comparison

| Method | Trainable Params | Best For | Notes |
|--------|-----------------|----------|-------|
| LoRA | <1% | General fine-tuning | Rank decomposition; all-rounder |
| QLoRA | <0.01% | LLM fine-tuning on consumer GPUs | 4-bit + LoRA; NF4 quantization |
| Prefix Tuning | ~0.01% | NLU tasks | Soft prompt vectors; no model mod |
| Prompt Tuning | ~0.01% | Large models (>10B) | Only soft embeddings |
| Adapters | ~0.5% | Multi-task | Bottleneck layers between blocks |
| BitFit | ~0.1% | Efficient bias | Only learns biases surprisingly effective |
| IA3 | tiny | Few-shot | Learned scaling vectors per layer |
| Full FT | 100% | Maximum quality | Expensive but best performance |

**Key insight**: QLoRA dominates when memory is limited but you still want LoRA-quality adaptation. Always start with QLoRA and only fall back to full FT if validation data is sufficient.